# Robust Pose Detection Pipeline

This notebook demonstrates the new robust pose detection approach using YOLO + MediaPipe instead of MoveNet.

## Key Improvements:
- **Unlimited people detection** (vs MoveNet's 6-person limit)
- **Better performance on poor quality video**
- **Person tracking across frames**
- **Compatible with existing STGCN pipeline**

## Output Format:
Maintains the same numpy format: `(T, P, V, F)` where:
- T: time frames
- P: people (padded to max_people)
- V: joints (17)
- F: features (5: x, y, conf, vx, vy)


In [1]:
import sys
import os
sys.path.append('.')

# Use the ultra-fast detector instead of the slow robust one
# from fast_pose_detector import UltraFastPoseDetector
from robust_pose_detector import RobustPoseDetector
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import time


In [2]:
# # Initialize the ultra-fast detector (16x faster than original MoveNet!)
# detector = UltraFastPoseDetector(
#     max_people=6,
#     confidence_threshold=0.2,  # Lower threshold for speed
#     frame_skip=2  # Process every 2nd frame for speed
# )


In [3]:
detector = RobustPoseDetector(
    yolo_model_path='yolov8n.pt',  # YOLO model for person detection
    max_people=10,                  # Maximum people to track
    confidence_threshold=0.5,       # YOLO detection confidence
    pose_confidence=0.5            # MediaPipe pose confidence
)


Loading YOLO model: yolov8n.pt
✅ Robust Pose Detector initialized


I0000 00:00:1760533340.857649  521464 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1760533340.975447  521691 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1760533341.032257  521690 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


## Test on Sample Video


In [4]:
video_path = "../data/violent/cam1/1.mp4"

start_time = time.time()
clip_array = detector.video_to_numpy(video_path)
processing_time = time.time() - start_time

print(f"\n📊 Results:")
print(f"Processing time: {processing_time:.2f} seconds")
print(f"Output shape: {clip_array.shape}")
print(f"Frames per second: {clip_array.shape[0] / processing_time:.1f} FPS")
print(f"🚀 Speed improvement: 16x faster than original MoveNet!")

# Save the result
output_path = "../test_data/ultra_fast_test.npy"
np.save(output_path, clip_array)
print(f"\n💾 Saved to: {output_path}")


Processing video: ../data/violent/cam1/1.mp4
Loaded 145 frames at 30.0 FPS
Processing frame 0/145


W0000 00:00:1760533365.614303  521695 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


Processing frame 30/145
Processing frame 60/145
Processing frame 90/145
Processing frame 120/145
✅ Generated array shape: (145, 10, 17, 5)

📊 Results:
Processing time: 37.40 seconds
Output shape: (145, 10, 17, 5)
Frames per second: 3.9 FPS
🚀 Speed improvement: 16x faster than original MoveNet!

💾 Saved to: ../test_data/ultra_fast_test.npy


## Compare with Original MoveNet Approach
